# CatBoost profundo: tratamiento de nulos, outliers y umbral

Este notebook documenta la ronda que produjo el mejor holdout agrupado. El modelo combina variables anidadas de MercadoLibre, atributos top-75 y 72 componentes SVD del texto.

**Advertencia:** el holdout se usa para comparar tratamientos; el umbral 0,49 se reporta como calibraci�n exploratoria, no como una estimaci�n independiente de generalizaci�n.

## Por qu� estos tratamientos

- **Nulos:** en CatBoost se conservan como categor�a expl�cita y se agregan conteos de ausencia; que falte `variations` o `shipping` puede ser informativo.
- **Outliers:** no se recortan autom�ticamente. Un precio o stock extremo puede representar una publicaci�n leg�tima. Se prueban flags IQR calculados solo con desarrollo, sin reemplazar el valor original.
- **Texto:** SVD reduce TF-IDF a se�ales densas y evita una matriz enorme de tokens.
- **Decisi�n:** se escanea el umbral de `P(used)` alrededor de 0,5 para compensar una asimetr�a residual de errores.

In [ ]:
# En una copia local del repo:
from ap2026q2_deep_features import build_deep_features, add_outlier_flags
# deep = build_deep_features(train, base_features)
# deep_test = build_deep_features(test, base_test_features)


In [ ]:
# Comparaci�n registrada en results/deep_feature_experiments.csv
import pandas as pd
scores = pd.read_csv('../results/deep_feature_experiments.csv')
display(scores.sort_values('accuracy', ascending=False))
scores.plot.bar(x='model', y=['accuracy','balanced_accuracy'], figsize=(12,4), rot=45, ylim=(.90,.95), grid=True)


In [ ]:
# Validaci�n de la submission final
sub = pd.read_csv('../../submissions/submission_deep_svd_depth8_threshold049.csv')
assert list(sub.columns) == ['ID','condition']
assert len(sub) == 30000
assert sub.ID.equals(pd.Series(range(1,30001), name='ID'))
assert set(sub.condition.unique()) <= {'new','used'}
display(sub.head())


## Resultado

El mejor resultado documentado es accuracy **0,93710** en el holdout agrupado, con balanced accuracy 0,93777 y macro F1 0,93687. El leaderboard visible tiene una referencia de 0,94880; la brecha restante requiere estudiar duplicados y se�ales de grupo/tiempo, no solo aumentar profundidad.